# Piracetam at the D2 Receptor — Full Pipeline (Day 18)

Dock → parameterise → merge → MD, all in one notebook. This is the **levetiracetam Day 11+12 protocol applied to piracetam**, with every Day 12 fix baked in from the start:

- NAGL charges (no AM1-BCC attempt)
- Placeholder box before GROMACS export
- `atom.atomic_number` (ParmEd version-safe)
- `MonteCarloMembraneBarostat` (semi-isotropic, tensionless)
- **Automatic heavy-atom mapping** — no hand-derived mapping table; works for any ligand via RDKit connectivity matching

**Piracetam:** 2-(2-oxopyrrolidin-1-yl)acetamide. C₁₁H₂₀N₂O₂, MW 212.29, 10 heavy atoms, 20 total. SMILES `NC(=O)CN1CCCC1=O`.

**Reused infrastructure (already on /workspace/d2_system/):**
| File | Role |
|---|---|
| `gmx_top.pkl` | D2 + DPPC ParmEd structure (CHARMM36) |
| `day9_final_equilibrated.xml` | Equilibrated D2 state (positions/box/velocities) |
| `day12_d2_equilibrated_protein.pdbqt` | Prepared receptor for docking |

**Outputs** use the `day14_briv_` prefix so nothing collides with the levetiracetam files.

**The scientific question:** piracetam is the original, lowest-potency racetam — it lacks both the ethyl tail of levetiracetam and the propyl group of brivaracetam, and it does NOT bind SV2A appreciably. Does it still reach the D2 pocket and the HIS393 anchor, or does the simpler structure fall away? Piracetam is the natural negative-leaning control for the series.

---
## Part 1 — Environment and inputs

In [ ]:
# Cell 1: Imports and versions
import sys, os, time, pickle
from pathlib import Path
import numpy as np

import openmm
from openmm import (XmlSerializer, LangevinMiddleIntegrator, MonteCarloMembraneBarostat,
                    Platform, CustomExternalForce, Vec3)
from openmm import unit as u
from openmm import app

import parmed as pmd
from openff.toolkit import Molecule, ForceField as OFFForceField

from rdkit import Chem
from rdkit.Chem import AllChem, rdDetermineBonds

print("OpenMM      :", openmm.__version__)
print("ParmEd      :", pmd.__version__)
import openff.toolkit, openff.interchange
print("OpenFF tk   :", openff.toolkit.__version__)
print("Interchange :", openff.interchange.__version__)

plats = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
assert "CUDA" in plats, f"CUDA not found; platforms = {plats}"
print("✓ CUDA available")

In [ ]:
# Cell 2: Working directory + required inputs
WORKDIR = Path("/workspace/d2_system")
os.chdir(WORKDIR)
print("CWD:", Path.cwd())

REQUIRED = {
    "gmx_top.pkl"                       : "D2 + DPPC ParmEd structure",
    "day9_final_equilibrated.xml"       : "Equilibrated D2 state",
    "day12_d2_equilibrated_protein.pdbqt": "Prepared receptor for docking",
}
missing = []
for name, desc in REQUIRED.items():
    p = WORKDIR / name
    if p.exists():
        print(f"  ✓ {name:38s} ({p.stat().st_size/1e6:6.1f} MB) — {desc}")
    else:
        print(f"  ✗ {name:38s} MISSING — {desc}"); missing.append(name)
if missing:
    raise FileNotFoundError(f"Missing inputs: {missing}")
print("\n✓ All inputs present")

# Output prefix for this ligand
PREFIX = "day18_pira"
LIGNAME = "PIR"          # 3-letter residue name used in trajectories
PIRACETAM_SMILES = "NC(=O)CN1CCCC1=O"   # achiral

---
## Part 2 — Build the piracetam ligand for docking

Built from SMILES (correct chemistry guaranteed), embedded in 3D, written to SDF and then PDBQT via Meeko. No PubChem download — the SMILES is the source of truth.

In [ ]:
# Cell 3: Build piracetam, sanity-check, write SDF
mol = Chem.MolFromSmiles(PIRACETAM_SMILES)
assert mol is not None, "SMILES failed to parse"
centers = Chem.FindMolChiralCenters(mol, useLegacyImplementation=False)
print(f"Formula      : {Chem.rdMolDescriptors.CalcMolFormula(mol)}")
print(f"Heavy atoms  : {mol.GetNumHeavyAtoms()} (expected 10)")
print(f"Chiral CIP   : {centers}  (expected none — piracetam is achiral)")
assert mol.GetNumHeavyAtoms() == 10

mol_h = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol_h, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol_h)
print(f"Total atoms  : {mol_h.GetNumAtoms()} (expected 20)")

# Write SDF for Meeko
LIG_SDF = WORKDIR / f"{PREFIX}_ligand.sdf"
Chem.MolToMolFile(mol_h, str(LIG_SDF))
print(f"✓ Wrote {LIG_SDF.name}")

In [ ]:
# Cell 4: Convert SDF → PDBQT with Meeko (for Vina)
# Meeko API differs slightly by version; this handles the common cases.
from meeko import MoleculePreparation
try:
    from meeko import PDBQTWriterLegacy
    _HAVE_WRITER = True
except Exception:
    _HAVE_WRITER = False

LIG_PDBQT = WORKDIR / f"{PREFIX}_ligand.pdbqt"

prep = MoleculePreparation()
setups = prep.prepare(mol_h)
setup = setups[0] if isinstance(setups, list) else setups

if _HAVE_WRITER:
    pdbqt_string, ok, err = PDBQTWriterLegacy.write_string(setup)
    assert ok, f"Meeko writer failed: {err}"
    LIG_PDBQT.write_text(pdbqt_string)
else:
    # older Meeko
    LIG_PDBQT.write_text(setup.write_pdbqt_string())

print(f"✓ Wrote {LIG_PDBQT.name} ({LIG_PDBQT.stat().st_size} bytes)")

---
## Part 3 — Dock piracetam to the equilibrated D2 pocket

Same pocket center and box as the levetiracetam re-dock, so the two ligands are compared in the identical site. Vina Python API, exhaustiveness 32, 9 poses.

In [ ]:
# Cell 5: Dock with AutoDock Vina
from vina import Vina

POCKET_CENTER = [53.806, 54.048, 62.877]   # same orthosteric pocket used for levetiracetam
BOX_SIZE      = [20.0, 20.0, 20.0]

v = Vina(sf_name="vina", cpu=0, verbosity=1)
v.set_receptor("day12_d2_equilibrated_protein.pdbqt")
v.set_ligand_from_file(str(LIG_PDBQT))
v.compute_vina_maps(center=POCKET_CENTER, box_size=BOX_SIZE)

print("Docking piracetam (exhaustiveness=32, 9 poses) ...")
t0 = time.time()
v.dock(exhaustiveness=32, n_poses=9)
energies = v.energies(n_poses=9)
print(f"Done in {time.time()-t0:.1f}s\n")
print("Pose scores (kcal/mol):")
for i, e in enumerate(energies):
    print(f"  Pose {i+1}: {e[0]:.3f}")
best_score = energies[0][0]
print(f"\n🎯 Best piracetam pose: {best_score:.3f} kcal/mol")
print(f"   (levetiracetam: -5.423 ; brivaracetam: -6.125 kcal/mol)")

DOCKED_PDBQT = WORKDIR / f"{PREFIX}_docked_poses.pdbqt"
v.write_poses(str(DOCKED_PDBQT), n_poses=9, overwrite=True)
print(f"✓ Saved {DOCKED_PDBQT.name}")

---
## Part 4 — Parameterise piracetam at its docked pose

This is the robust, automatic version of what was painful for levetiracetam. The docked pose's heavy-atom coordinates are read from the best Vina pose, then mapped onto a clean SMILES-built molecule by **RDKit connectivity matching** — no hand-derived mapping table. Charges via NAGL.

In [ ]:
# Cell 6: Parse best docked pose heavy-atom coordinates from the PDBQT
def parse_pdbqt_first_pose_heavy(pdbqt_path):
    coords, elems = [], []
    in_model1 = True
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith("MODEL") and "2" in line.split()[-1:]:
                break  # stop after first model
            if line.startswith("ENDMDL"):
                break
            if line.startswith(("ATOM", "HETATM")):
                name = line[12:16].strip()
                if name.startswith("H"):     # skip polar H's that Meeko kept
                    continue
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                # element from columns 76-78 if present, else from atom name
                el = line[76:78].strip()
                if not el or el in ("A",):   # 'A' = aromatic carbon in PDBQT
                    el = "C"
                # PDBQT autodock types: map back to element
                el = {"OA":"O","NA":"N","HD":"H","SA":"S","C":"C","N":"N","O":"O"}.get(el, el[0])
                if el == "H":
                    continue
                coords.append([x, y, z]); elems.append(el)
    return np.array(coords), elems

docked_coords, docked_elems = parse_pdbqt_first_pose_heavy(DOCKED_PDBQT)
print(f"Docked heavy atoms parsed: {len(docked_coords)} {docked_elems}")
assert len(docked_coords) == 10, f"Expected 10 heavy atoms, got {len(docked_coords)}"
print(f"Centroid (Å): {docked_coords.mean(axis=0)}")

In [ ]:
# Cell 7: AUTOMATIC heavy-atom mapping (connectivity match) — no manual table
def connectivity_only(m):
    em = Chem.RWMol(m)
    for b in em.GetBonds():
        b.SetBondType(Chem.BondType.SINGLE); b.SetIsAromatic(False)
    for a in em.GetAtoms():
        a.SetIsAromatic(False); a.SetFormalCharge(0); a.SetNoImplicit(True)
    return em.GetMol()

# Build a heavy-atom-only molecule from the docked coordinates, infer connectivity
xyz = f"{len(docked_coords)}\n\n"
for el,(x,y,z) in zip(docked_elems, docked_coords):
    xyz += f"{el} {x:.4f} {y:.4f} {z:.4f}\n"
docked_mol = Chem.MolFromXYZBlock(xyz)
rdDetermineBonds.DetermineConnectivity(docked_mol)
print(f"Docked mol: {docked_mol.GetNumAtoms()} atoms, {docked_mol.GetNumBonds()} bonds inferred")

# Clean template from SMILES (heavy atoms only)
template = Chem.MolFromSmiles(PIRACETAM_SMILES)
assert template.GetNumHeavyAtoms() == len(docked_coords)

# Match template → docked by connectivity (ignores bond orders)
match = connectivity_only(docked_mol).GetSubstructMatch(connectivity_only(template))
assert len(match) == 10, (
    f"Automatic mapping failed (got {len(match)} of 15). "
    "Bond inference on the docked pose may be ambiguous — send Claude the docked PDBQT."
)
# Element-consistency check
for ti in range(len(match)):
    assert template.GetAtomWithIdx(ti).GetSymbol() == docked_elems[match[ti]], \
        f"Element mismatch at template atom {ti}"
print(f"✓ Automatic mapping succeeded: 10/10 atoms, element-consistent")
print(f"  Mapping (template_i → docked_i): {match}")

In [ ]:
# Cell 8: Transplant docked coords onto a clean ligand, relax H's, NAGL charges
template_h = Chem.AddHs(template)
AllChem.EmbedMolecule(template_h, randomSeed=42)
AllChem.MMFFOptimizeMolecule(template_h)
conf = template_h.GetConformer()

# Heavy atoms of template_h are indices 0..14 (Hs come after AddHs)
template_heavy = [a.GetIdx() for a in template_h.GetAtoms() if a.GetAtomicNum() > 1]
assert len(template_heavy) == 10

# template atom ti (0..14) → docked coordinate row match[ti]
for ti, mol_idx in enumerate(template_heavy):
    x, y, z = docked_coords[match[ti]]
    conf.SetAtomPosition(mol_idx, (float(x), float(y), float(z)))

# Constrained minimise: fix heavy atoms, relax hydrogens
mp = AllChem.MMFFGetMoleculeProperties(template_h)
ff = AllChem.MMFFGetMoleculeForceField(template_h, mp)
for i in template_heavy:
    ff.AddFixedPoint(i)
ff.Minimize(maxIts=500)
print("✓ Hydrogens relaxed around fixed docked heavy atoms")

# Heavy-atom bond-length sanity check
for b in template_h.GetBonds():
    a1, a2 = b.GetBeginAtom(), b.GetEndAtom()
    if a1.GetAtomicNum()==1 or a2.GetAtomicNum()==1: continue
    p1 = conf.GetAtomPosition(a1.GetIdx()); p2 = conf.GetAtomPosition(a2.GetIdx())
    d = ((p1.x-p2.x)**2+(p1.y-p2.y)**2+(p1.z-p2.z)**2)**0.5
    if not (1.0 < d < 1.8):
        print(f"  ⚠ {a1.GetSymbol()}{a1.GetIdx()}–{a2.GetSymbol()}{a2.GetIdx()}: {d:.2f} Å")
print("✓ Heavy-atom bond lengths OK")

# → OpenFF molecule + NAGL charges
ligand_off = Molecule.from_rdkit(template_h)
from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper
NAGLToolkitWrapper().assign_partial_charges(
    molecule=ligand_off, partial_charge_method="openff-gnn-am1bcc-0.1.0-rc.3.pt")
print(f"✓ NAGL charges assigned (sum = {sum(c.m for c in ligand_off.partial_charges):.3f} e)")
print(f"  SMILES (mapped): {ligand_off.to_smiles()}")
assert ligand_off.n_atoms == 20

In [ ]:
# Cell 9: OpenFF → Interchange → GROMACS → ParmEd
print("Parameterising with OpenFF Sage 2.1.0 ...")
off_ff = OFFForceField("openff-2.1.0.offxml")
interchange = off_ff.create_interchange(
    topology=ligand_off.to_topology(), charge_from_molecules=[ligand_off])

# Placeholder box (required for GROMACS export; merge overrides it)
from openff.units import unit as off_unit
interchange.box = np.eye(3) * 5.0 * off_unit.nanometer

LIG_TOP = WORKDIR / f"_tmp_{PREFIX}.top"
LIG_GRO = WORKDIR / f"_tmp_{PREFIX}.gro"
interchange.to_top(str(LIG_TOP)); interchange.to_gro(str(LIG_GRO))

ligand_struct = pmd.load_file(str(LIG_TOP), xyz=str(LIG_GRO))
# Force coordinates to docked pose (heavy atoms from docking; Hs relaxed)
ligand_struct.coordinates = np.array(
    [list(conf.GetAtomPosition(i)) for i in range(template_h.GetNumAtoms())])
for r in ligand_struct.residues:
    r.name = LIGNAME
print(f"✓ Ligand ParmEd structure: {len(ligand_struct.atoms)} atoms, "
      f"centroid {ligand_struct.coordinates.mean(axis=0)}")

---
## Part 5 — Load equilibrated D2 and merge

In [ ]:
# Cell 10: Load D2 ParmEd structure + equilibrated state
with open("gmx_top.pkl", "rb") as f:
    d2_struct = pickle.load(f)
print(f"D2 structure: {len(d2_struct.atoms):,} atoms")

with open("day9_final_equilibrated.xml") as f:
    equil_state = XmlSerializer.deserialize(f.read())
equil_pos = equil_state.getPositions(asNumpy=True)
equil_box = equil_state.getPeriodicBoxVectors(asNumpy=True)
assert len(equil_pos) == len(d2_struct.atoms)

d2_struct.coordinates = equil_pos.value_in_unit(u.angstrom)
d2_struct.box = [equil_box[0][0].value_in_unit(u.angstrom),
                 equil_box[1][1].value_in_unit(u.angstrom),
                 equil_box[2][2].value_in_unit(u.angstrom), 90.0, 90.0, 90.0]
print(f"✓ D2 updated to equilibrated state; box (Å) = {d2_struct.box}")

In [ ]:
# Cell 11: Merge D2 + piracetam
combined = d2_struct + ligand_struct
combined.box = d2_struct.box

n_d2, n_lig = len(d2_struct.atoms), len(ligand_struct.atoms)
print(f"D2 {n_d2:,} + ligand {n_lig} = combined {len(combined.atoms):,}")
assert len(combined.atoms) == n_d2 + n_lig

from collections import Counter
rc = Counter(r.name for r in combined.residues)
assert rc.get(LIGNAME, 0) == 1, f"{LIGNAME} not found after merge"
print(f"✓ Merge OK, {LIGNAME} present. Box (Å) = {combined.box}")

---
## Part 6 — Build the OpenMM system

In [ ]:
# Cell 12: createSystem + membrane barostat
print("Creating OpenMM system ...")
t0 = time.time()
system = combined.createSystem(
    nonbondedMethod=app.PME, nonbondedCutoff=1.0*u.nanometer,
    constraints=app.HBonds, rigidWater=True, removeCMMotion=True,
    ewaldErrorTolerance=5e-4)
print(f"  Built in {time.time()-t0:.1f}s; {system.getNumParticles():,} particles")
for i in range(system.getNumForces()):
    print(f"    {i}: {system.getForce(i).__class__.__name__}")

barostat = MonteCarloMembraneBarostat(
    1.0*u.bar, 0.0*u.bar*u.nanometer, 310*u.kelvin,
    MonteCarloMembraneBarostat.XYIsotropic, MonteCarloMembraneBarostat.ZFree, 25)
system.addForce(barostat)
print("✓ Membrane barostat added (XYIsotropic, ZFree, 310 K, 1 bar, γ=0)")

In [ ]:
# Cell 13: Simulation object on CUDA
integrator = LangevinMiddleIntegrator(310*u.kelvin, 1.0/u.picosecond, 0.002*u.picosecond)
platform = Platform.getPlatformByName("CUDA")
simulation = app.Simulation(combined.topology, system, integrator, platform,
                            {"Precision": "mixed", "DeviceIndex": "0"})
simulation.context.setPositions(combined.positions)
box_nm = np.array(combined.box[:3]) / 10.0
simulation.context.setPeriodicBoxVectors(
    Vec3(box_nm[0],0,0)*u.nanometer, Vec3(0,box_nm[1],0)*u.nanometer, Vec3(0,0,box_nm[2])*u.nanometer)
print(f"✓ Simulation ready on {simulation.context.getPlatform().getName()}, "
      f"{system.getNumParticles():,} atoms")

---
## Part 7 — Minimise and equilibrate

Watch the minimisation energy: a large positive value before (clashes from ligand insertion) dropping to ≈ −1.4 × 10⁶ kJ/mol after is the healthy signature you saw for levetiracetam.

In [ ]:
# Cell 14: Energy minimisation
e0 = simulation.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(u.kilojoule_per_mole)
print(f"PE before: {e0:,.1f} kJ/mol")
print("Minimising ...")
t0 = time.time()
simulation.minimizeEnergy(maxIterations=5000, tolerance=10*u.kilojoule_per_mole/u.nanometer)
e1 = simulation.context.getState(getEnergy=True).getPotentialEnergy().value_in_unit(u.kilojoule_per_mole)
print(f"PE after : {e1:,.1f} kJ/mol  (Δ={e1-e0:,.1f}) in {time.time()-t0:.1f}s")
simulation.saveState(f"{PREFIX}_minimized.xml")
# Save a topology PDB for trajectory analysis later (records the BRV residue)
with open(f"{PREFIX}_minimized.pdb","w") as f:
    app.PDBFile.writeFile(combined.topology,
        simulation.context.getState(getPositions=True).getPositions(), f)
print(f"✓ Saved {PREFIX}_minimized.xml and .pdb")

In [ ]:
# Cell 15: Restrain ligand heavy atoms, then 1 ns restrained NPT
lig_heavy = [a.idx for a in combined.atoms if a.residue.name == LIGNAME and a.atomic_number > 1]
print(f"Ligand heavy atoms: {len(lig_heavy)} (expected 10)")
assert len(lig_heavy) == 10

restraint = CustomExternalForce("0.5*k*periodicdistance(x,y,z,x0,y0,z0)^2")
restraint.addGlobalParameter("k", 1000.0*u.kilojoule_per_mole/u.nanometer**2)
for p in ("x0","y0","z0"): restraint.addPerParticleParameter(p)
pos = simulation.context.getState(getPositions=True).getPositions(asNumpy=True)
for idx in lig_heavy:
    p = pos[idx].value_in_unit(u.nanometer); restraint.addParticle(idx,[p[0],p[1],p[2]])
ridx = system.addForce(restraint)
simulation.context.reinitialize(preserveState=True)

simulation.context.setVelocitiesToTemperature(310*u.kelvin)
simulation.reporters.clear()
simulation.reporters.append(app.StateDataReporter(sys.stdout, 25000,
    step=True, time=True, temperature=True, volume=True, speed=True, separator=" | "))
print("Equilibration 1 (restrained, 1 ns) ...")
t0=time.time(); simulation.step(500_000)
print(f"✓ done in {(time.time()-t0)/60:.1f} min")
simulation.saveState(f"{PREFIX}_equil1.xml")

In [ ]:
# Cell 16: Remove restraints, 1 ns unrestrained NPT
system.removeForce(ridx)
simulation.context.reinitialize(preserveState=True)
simulation.reporters.clear()
simulation.reporters.append(app.StateDataReporter(sys.stdout, 25000,
    step=True, time=True, temperature=True, volume=True, speed=True, separator=" | "))
print("Equilibration 2 (unrestrained, 1 ns) ...")
t0=time.time(); simulation.step(500_000)
print(f"✓ done in {(time.time()-t0)/60:.1f} min")
simulation.saveState(f"{PREFIX}_equil2.xml")
simulation.saveCheckpoint(f"{PREFIX}_equil2.chk")

In [ ]:
# Cell 17: Quick pre-production check — is piracetam in the pocket?
import mdtraj as mdt
st = simulation.context.getState(getPositions=True)
with open(f"{PREFIX}_postequil2.pdb","w") as f:
    app.PDBFile.writeFile(combined.topology, st.getPositions(), f)
snap = mdt.load(f"{PREFIX}_postequil2.pdb")
lig = snap.topology.select(f"resname {LIGNAME}")
prot = snap.topology.select("protein and not type H")
pairs = np.array([[l,p] for l in lig for p in prot])
dmin = mdt.compute_distances(snap, pairs)[0].min()*10
print(f"Min ligand–protein distance: {dmin:.2f} Å  (<5 Å = in pocket)")
# HIS393 (sim HIS214) check — does piracetam also reach the lev anchor?
his_ne2 = snap.topology.select("resid 213 and name NE2")  # resid 213 (0-indexed) = HIS214
if len(his_ne2):
    lig_polar = snap.topology.select(f"resname {LIGNAME} and (type O or type N)")
    ph = np.array([[l,h] for l in lig_polar for h in his_ne2])
    print(f"Min ligand→HIS214(NE2) distance: {mdt.compute_distances(snap,ph)[0].min()*10:.2f} Å")

---
## Part 8 — Production MD (50 ns)

~6 hours on an RTX 4090. Frames every 50 ps. Disconnect-safe — the GPU keeps running if you close the browser, as long as the pod stays up.

In [ ]:
# Cell 18: 50 ns production
PROD_NS = 50.0
PROD_STEPS = int(PROD_NS * 500_000)
simulation.reporters.clear()
simulation.reporters.append(app.DCDReporter(f"{PREFIX}_production.dcd", 25000))
simulation.reporters.append(app.CheckpointReporter(f"{PREFIX}_production.chk", 500_000))
simulation.reporters.append(app.StateDataReporter(f"{PREFIX}_production.log", 5000,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True,
    temperature=True, volume=True, density=True, speed=True, remainingTime=True,
    totalSteps=PROD_STEPS, separator=","))
simulation.reporters.append(app.StateDataReporter(sys.stdout, 100_000,
    step=True, time=True, temperature=True, density=True, speed=True,
    remainingTime=True, totalSteps=PROD_STEPS, separator=" | "))

print(f"Production {PROD_NS} ns starting {time.strftime('%H:%M:%S')} ...")
t0=time.time(); simulation.step(PROD_STEPS)
print(f"✓ done in {(time.time()-t0)/3600:.2f} h")
simulation.saveState(f"{PREFIX}_production_final.xml")
simulation.saveCheckpoint(f"{PREFIX}_production.chk")
print(f"✓ Saved {PREFIX}_production_final.xml")

In [ ]:
import os, numpy as np, mdtraj as mdt, matplotlib.pyplot as plt
from collections import Counter
os.chdir("/workspace/d2_system")
PREFIX, LIGNAME = "day18_pira", "PIR"

traj = mdt.load(f"{PREFIX}_production.dcd", top=f"{PREFIX}_minimized.pdb")
print(f"Loaded {traj.n_frames} frames ({traj.n_frames*0.05:.0f} ns)")

ref = traj[0]
ca = traj.topology.select("protein and name CA")
lig_heavy = traj.topology.select(f"resname {LIGNAME} and not type H")
traj.superpose(ref, atom_indices=ca)
rmsd = mdt.rmsd(traj, ref, atom_indices=lig_heavy)*10
t_ns = np.arange(traj.n_frames)*0.05

# HIS393 (sim HIS214 = resid 213)
his_ne2 = traj.topology.select("resid 213 and name NE2")
lig_polar = traj.topology.select(f"resname {LIGNAME} and (type O or type N)")
pairs = np.array([[l,h] for l in lig_polar for h in his_ne2])
dmin = mdt.compute_distances(traj, pairs).min(axis=1)*10
hbond_pct = 100*(dmin < 3.5).mean()

# contacts
prot_heavy = traj.topology.select("protein and not type H")
contacts = mdt.compute_neighbors(traj, 0.4, query_indices=lig_heavy, haystack_indices=prot_heavy)
ncontact = np.array([len(c) for c in contacts])

# per-residue contact frequency (to see WHERE it bound)
atom2res = {}
for r in traj.topology.residues:
    if r.is_protein:
        for a in r.atoms:
            if a.element.symbol != "H": atom2res[a.index] = r
counts = Counter()
for frame in contacts:
    for r in {atom2res[a] for a in frame if a in atom2res}: counts[r] += 1

fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot(t_ns, rmsd, lw=1); ax[0].axhline(3, ls="--", c="gray")
ax[0].set_xlabel("Time (ns)"); ax[0].set_ylabel("Ligand RMSD (A)"); ax[0].set_title("Piracetam RMSD")
ax[1].plot(t_ns, dmin, lw=1, c="#d62728"); ax[1].axhline(3.5, ls="--", c="gray")
ax[1].set_xlabel("Time (ns)"); ax[1].set_ylabel("min dist (A)"); ax[1].set_title("Piracetam to HIS393 NE2")
plt.tight_layout(); plt.savefig(f"{PREFIX}_quicklook.png", dpi=150); plt.show()

print(f"\n=== THREE-RACETAM COMPARISON (D2) ===")
print(f"                    PIRACETAM   levetiracetam   brivaracetam")
print(f"Vina score (kcal):    -4.734        -5.423         -6.125")
print(f"Ligand RMSD mean :    {rmsd.mean():5.2f} A       0.98 A         1.27 A")
print(f"% frames <3 A    :   {100*(rmsd<3).mean():5.1f}%       100.0%         100.0%")
print(f"HIS393 H-bond %  :   {hbond_pct:5.1f}%        89.0%          3.2%")
print(f"Mean contacts    :    {ncontact.mean():5.1f}        31.2           14.2")
print(f"\nPiracetam top 10 contact residues:")
for r,c in counts.most_common(10):
    print(f"  {r.name}{r.resSeq:<4} {100*c/traj.n_frames:5.1f}%")

In [ ]:
import os, numpy as np, mdtraj as mdt
from collections import Counter
os.chdir("/workspace/d2_system")
PREFIX, LIGNAME = "day18_pira", "PIR"

traj = mdt.load(f"{PREFIX}_production.dcd", top=f"{PREFIX}_minimized.pdb")
lig_heavy = traj.topology.select(f"resname {LIGNAME} and not type H")
prot_heavy = traj.topology.select("protein and not type H")

atom2res = {}
for r in traj.topology.residues:
    if r.is_protein:
        for a in r.atoms:
            if a.element.symbol != "H":
                atom2res[a.index] = r

def contact_profile(sub):
    contacts = mdt.compute_neighbors(sub, 0.4, query_indices=lig_heavy, haystack_indices=prot_heavy)
    counts = Counter()
    for frame in contacts:
        for r in {atom2res[a] for a in frame if a in atom2res}:
            counts[r] += 1
    return counts, sub.n_frames

early, ne = contact_profile(traj[:400])    # 0-20 ns
late,  nl = contact_profile(traj[400:])    # 20-50 ns

print("=== PIRACETAM EARLY (0-20 ns) top contacts ===")
for r,c in early.most_common(12):
    print(f"  {r.name}{r.resSeq:<4} {100*c/ne:5.1f}%")
print("\n=== PIRACETAM LATE (20-50 ns) top contacts ===")
for r,c in late.most_common(12):
    print(f"  {r.name}{r.resSeq:<4} {100*c/nl:5.1f}%")

# Quantify how stable the contact SET is: overlap of top-10 early vs late
top_early = {r for r,_ in early.most_common(10)}
top_late  = {r for r,_ in late.most_common(10)}
overlap = len(top_early & top_late)
print(f"\nTop-10 residue overlap early vs late: {overlap}/10")
print("(High overlap = same residues throughout = stable specific binding)")
print("(Low overlap = shifting partners = drifting/non-specific)")

---
## Part 9 — Quick analysis (compare to levetiracetam)

The two headline numbers to compare against levetiracetam: ligand RMSD stability and HIS393 H-bond persistence. Full per-residue and contact analysis goes in the comparison notebook once all three racetams are done.

In [ ]:
# Cell 19: Ligand RMSD + HIS393 H-bond persistence
import mdtraj as mdt, matplotlib.pyplot as plt
traj = mdt.load(f"{PREFIX}_production.dcd", top=f"{PREFIX}_minimized.pdb")
print(f"Loaded {traj.n_frames} frames ({traj.n_frames*0.05:.0f} ns)")

ref = traj[0]
ca = traj.topology.select("protein and name CA")
lig_heavy = traj.topology.select(f"resname {LIGNAME} and not type H")
traj.superpose(ref, atom_indices=ca)
rmsd = mdt.rmsd(traj, ref, atom_indices=lig_heavy)*10
t_ns = np.arange(traj.n_frames)*0.05

# HIS393 (sim HIS214 = resid 213) NE2 to ligand polar atoms
his_ne2 = traj.topology.select("resid 213 and name NE2")
lig_polar = traj.topology.select(f"resname {LIGNAME} and (type O or type N)")
pairs = np.array([[l,h] for l in lig_polar for h in his_ne2])
dmin = mdt.compute_distances(traj, pairs).min(axis=1)*10
hbond_pct = 100*(dmin < 3.5).mean()

fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot(t_ns, rmsd, lw=1); ax[0].axhline(3, ls="--", c="gray")
ax[0].set_xlabel("Time (ns)"); ax[0].set_ylabel("Ligand RMSD (Å)")
ax[0].set_title("Piracetam RMSD vs start")
ax[1].plot(t_ns, dmin, lw=1, c="#d62728"); ax[1].axhline(3.5, ls="--", c="gray")
ax[1].set_xlabel("Time (ns)"); ax[1].set_ylabel("min dist (Å)")
ax[1].set_title("Piracetam → HIS393(NE2)")
plt.tight_layout(); plt.savefig(f"{PREFIX}_quicklook.png", dpi=150); plt.show()

print(f"\n=== PIRACETAM vs LEVETIRACETAM ===")
print(f"Ligand RMSD mean : {rmsd.mean():.2f} Å   (lev: 0.98)")
print(f"Ligand RMSD max  : {rmsd.max():.2f} Å   (lev: 1.96)")
print(f"% frames <3 Å    : {100*(rmsd<3).mean():.1f}%  (lev: 100.0)")
print(f"HIS393 H-bond %  : {hbond_pct:.1f}%  (lev: 89.0)")
print(f"Vina best score  : {best_score:.3f}  (lev: -5.423 ; briv: -6.125)")

---
## Outputs

| File | Contents |
|---|---|
| `day14_briv_ligand.sdf/.pdbqt` | Built ligand for docking |
| `day14_briv_docked_poses.pdbqt` | Vina poses |
| `day14_briv_minimized.xml/.pdb` | Post-minimisation state + topology |
| `day14_briv_equil2.xml/.chk` | After equilibration |
| `day14_briv_production.dcd` | 50 ns trajectory |
| `day14_briv_production.log` | Energy/T/density log |
| `day14_briv_production_final.xml` | Final state |
| `day14_briv_quicklook.png` | RMSD + HIS393 comparison plot |

**This completes the three-racetam series at D2.** Next: the comparison notebook — load all three trajectories (levetiracetam, brivaracetam, piracetam), compute HIS393 H-bond persistence and ligand RMSD for each, and build the comparison table and figure for the paper.

In [ ]:
REPLICATE_ID = 2
PROD_NS      = 50.0
PROD_STEPS   = int(PROD_NS * 500_000)

tag = f"rep{REPLICATE_ID}"

with open(f"{PREFIX}_equil2.xml") as f:
    st = XmlSerializer.deserialize(f.read())

simulation.context.setPeriodicBoxVectors(*st.getPeriodicBoxVectors())
simulation.context.setPositions(st.getPositions())
simulation.context.setVelocitiesToTemperature(310 * u.kelvin, 3000 + REPLICATE_ID)
simulation.currentStep = 0
print(f"{PREFIX} replicate {REPLICATE_ID}: seed {3000+REPLICATE_ID}")

simulation.reporters.clear()
simulation.reporters.append(app.DCDReporter(f"{PREFIX}_production_{tag}.dcd", 25000))
simulation.reporters.append(app.CheckpointReporter(f"{PREFIX}_production_{tag}.chk", 500_000))
simulation.reporters.append(app.StateDataReporter(
    f"{PREFIX}_production_{tag}.log", 5_000,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True, density=True,
    speed=True, remainingTime=True, totalSteps=PROD_STEPS, separator=","))
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, 100_000, step=True, time=True, temperature=True,
    density=True, speed=True, remainingTime=True,
    totalSteps=PROD_STEPS, separator=" | "))

t0 = time.time()
simulation.step(PROD_STEPS)
elapsed = (time.time() - t0) / 3600
print(f"\n✓ {tag} done in {elapsed:.2f} h ({PROD_NS/elapsed:.1f} ns/day)")
simulation.saveState(f"{PREFIX}_production_{tag}_final.xml")

In [ ]:
REPLICATE_ID = 3
PROD_NS      = 50.0
PROD_STEPS   = int(PROD_NS * 500_000)

tag = f"rep{REPLICATE_ID}"

with open(f"{PREFIX}_equil2.xml") as f:
    st = XmlSerializer.deserialize(f.read())

simulation.context.setPeriodicBoxVectors(*st.getPeriodicBoxVectors())
simulation.context.setPositions(st.getPositions())
simulation.context.setVelocitiesToTemperature(310 * u.kelvin, 3000 + REPLICATE_ID)
simulation.currentStep = 0
print(f"{PREFIX} replicate {REPLICATE_ID}: seed {3000+REPLICATE_ID}")

simulation.reporters.clear()
simulation.reporters.append(app.DCDReporter(f"{PREFIX}_production_{tag}.dcd", 25000))
simulation.reporters.append(app.CheckpointReporter(f"{PREFIX}_production_{tag}.chk", 500_000))
simulation.reporters.append(app.StateDataReporter(
    f"{PREFIX}_production_{tag}.log", 5_000,
    step=True, time=True, potentialEnergy=True, kineticEnergy=True,
    totalEnergy=True, temperature=True, volume=True, density=True,
    speed=True, remainingTime=True, totalSteps=PROD_STEPS, separator=","))
simulation.reporters.append(app.StateDataReporter(
    sys.stdout, 100_000, step=True, time=True, temperature=True,
    density=True, speed=True, remainingTime=True,
    totalSteps=PROD_STEPS, separator=" | "))

t0 = time.time()
simulation.step(PROD_STEPS)
elapsed = (time.time() - t0) / 3600
print(f"\n✓ {tag} done in {elapsed:.2f} h ({PROD_NS/elapsed:.1f} ns/day)")
simulation.saveState(f"{PREFIX}_production_{tag}_final.xml")